# SimMOF End-to-End Demo

This notebook demonstrates how SimMOF converts natural-language MOF simulation requests into structured workflows.

We will cover:
1. a simple end-to-end example,
2. a query that requires clarification,
3. patching user-provided simulation input.

In [ ]:
import json

from query.agent import analyze_mof_query
from working.agent import WorkingAgent

from Zeopp.agent import ZeoppAgent
from LAMMPS.agent import LAMMPSAgent
from response.agent import ResponseAgent
from RASPA.agent import RASPAAgent
from VASP.agent import VASPAgent
from screening.agent import ScreeningAgent
from analysis.agent import AnalysisAgent

from main import patch_simulation_input_with_user_reply
from config import LLM_DEFAULT

In [ ]:
agents = {
    "ZeoppAgent": ZeoppAgent(),
    "LAMMPSAgent": LAMMPSAgent(),
    "ResponseAgent": ResponseAgent(),
    "RASPAAgent": RASPAAgent(),
    "VASPAgent": VASPAgent(),
    "ScreeningAgent": ScreeningAgent(),
    "AnalysisAgent": AnalysisAgent(),
}

In [ ]:
def show_result(title, obj):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    if isinstance(obj, str):
        print(obj)
    else:
        print(json.dumps(obj, ensure_ascii=False, indent=2))


def build_workflow(user_query, agents):
    bundle = analyze_mof_query(user_query)

    if not bundle or not bundle.get("queries"):
        return {
            "ok": False,
            "error": "Query parsing failed.",
            "bundle": bundle,
        }

    parsed = bundle["queries"]
    analysis_enabled = bundle.get("analysis_enabled", False)
    simulation_input = bundle.get("simulation_input")

    wa = WorkingAgent(
        parsed,
        analysis_enabled=analysis_enabled,
        agents=agents,
        simulation_input=simulation_input,
    )

    plans = wa.plan()

    return {
        "ok": True,
        "bundle": bundle,
        "parsed": parsed,
        "analysis_enabled": analysis_enabled,
        "simulation_input": simulation_input,
        "plans": [p.model_dump() for p in plans],
        "worker": wa,
    }

## How to read this notebook

For each example, we will inspect:
- the original user query,
- the parsed query bundle,
- whether clarification is needed,
- the planned workflow,
- and optionally the execution behavior.

## Example 1 — Simple Zeo++ workflow

In this example, the user asks for the surface area of UiO-66.

This is a simple structural-property query that should map naturally to a Zeo++-based workflow,
making it a good introductory example for new users.

In [ ]:
user_query_1 = "I want to calculate the surface area of UiO-66"
print(user_query_1)

In [ ]:
demo1 = build_workflow(user_query_1, agents)

show_result("Parsed bundle", demo1["bundle"])
show_result("Parsed queries", demo1["parsed"])
show_result("Planned workflows", demo1["plans"])

In [ ]:
# Uncomment the lines below if you want to run the full workflow.
# Depending on your environment, this may trigger the actual Zeo++-based execution.

results_1 = await demo1["worker"].run_async()
show_result("Execution results", results_1)

## Example 2 — Query requiring clarification

Now we test a more ambiguous request:

`Calculate adsorption in HKUST-1`

This query is incomplete because it does not specify key simulation conditions such as:
- the adsorbate species,
- temperature,
- pressure,
- or the exact adsorption-related property of interest.

In this example, we show how SimMOF detects the missing information,
asks for clarification, and then replans the workflow after the user provides additional conditions.

In [ ]:
user_query_2 = "Calculate adsorption in HKUST-1"
print(user_query_2)

In [ ]:
demo2 = build_workflow(user_query_2, agents)
show_result("Initial parsed bundle", demo2["bundle"])

In [ ]:
print("needs_clarification:", demo2["bundle"].get("needs_clarification", False))
print("clarification_question:")
print(demo2["bundle"].get("clarification_question", "No clarification needed."))

In [ ]:
user_reply_2 = "Use CO2 at 298 K and 1 bar."

refined_query_2 = (
    user_query_2.strip()
    + "\n"
    + f"Additional user-provided conditions: {user_reply_2}"
)

print(refined_query_2)

In [ ]:
demo2_refined = build_workflow(refined_query_2, agents)

show_result("Refined parsed bundle", demo2_refined["bundle"])
show_result("Refined parsed queries", demo2_refined["parsed"])
show_result("Refined planned workflows", demo2_refined["plans"])

In [ ]:
# Uncomment the lines below if you want to run the full workflow.
# Depending on your environment, this may trigger the actual RASPA-based execution.

results_2 = await demo2_refined["worker"].run_async()
show_result("Execution results", results_2)

## Example 3 — Reusing user-provided simulation input (reproduce mode)

In this example, the user directly provides a simulation input snippet.

SimMOF will:
1. detect the provided input,
2. check whether it matches the requested task,
3. and allow the user to reuse it as-is.

This corresponds to a reproduce-style workflow,
where existing simulation inputs are reused instead of being regenerated.

In [ ]:
user_query_3 = """I want to calculate CO2 uptake in UiO-66.

Here is my RASPA input:

SimulationType                MonteCarlo
NumberOfCycles               2000
NumberOfInitializationCycles 1000
PrintEvery                   100
Forcefield                   UFF
FrameworkName                UiO-66
ExternalTemperature          298.0
ExternalPressure             100000
Component 0 MoleculeName     CO2
            MoleculeDefinition TraPPE
            TranslationProbability 1.0
            RotationProbability    1.0
            ReinsertionProbability 1.0
"""
print(user_query_3)

In [ ]:
demo3 = build_workflow(user_query_3, agents)

show_result("Parsed bundle", demo3["bundle"])
show_result("Parsed queries", demo3["parsed"])
show_result("Simulation input", demo3["simulation_input"])

In [ ]:
print("simulation_input_status:", demo3["bundle"]["simulation_input_status"])
print("simulation_input_message:")
print(demo3["bundle"]["simulation_input_message"] or "(no issues detected)")

In [ ]:
user_decision_3 = "KEEP"   # or "REGENERATE" or a correction instruction
print("User decision:", user_decision_3)

In [ ]:
bundle3 = demo3["bundle"]

if bundle3["simulation_input_status"] == "needs_user_confirmation":
    if user_decision_3.upper() == "KEEP":
        print("Keeping the user-provided simulation input as-is.")
        results_3 = await demo3["worker"].run_async()
        show_result("Execution results", results_3)

    elif user_decision_3.upper() == "REGENERATE":
        print("Discarding the user-provided simulation input and regenerating from scratch.")

        regenerated_demo3 = build_workflow(
            demo3["bundle"]["queries"][0]["QueryText"],
            agents,
        )

        results_3 = await regenerated_demo3["worker"].run_async()
        show_result("Execution results", results_3)

    else:
        print("Applying a user-requested correction to the provided simulation input...")

        patched_simulation_input = patch_simulation_input_with_user_reply(
            simulation_input=bundle3["simulation_input"],
            user_reply=user_decision_3,
            parsed_queries=demo3["parsed"],
            llm=LLM_DEFAULT,
        )

        show_result("Patched simulation input", patched_simulation_input)
else:
    print("No confirmation needed. Running directly...")
    results_3 = await demo3["worker"].run_async()
    show_result("Execution results", results_3)